In [1]:
from openai import OpenAI
import gradio as gr
import json

c:\Users\apoor\OneDrive\Desktop\git\GENAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [3]:
system_prompt = """You are helpful assistant for an Arline called FlightAI.
Give short, courteous answes, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def airline_chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content": message}]
    response = openai.chat.completions.create(model="llama3.1:latest", messages=messages)
    return response.choices[0].message.content
gr.ChatInterface(fn=airline_chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
ticket_price = {"London":"$1100", "Paris" :"$745", "Zurich":"$975", "Delhi": "760"}
def ticket_price_query(destination_city):
    print(f"Tool Called for city : {destination_city}")
    price = ticket_price.get(destination_city, "Unknown city")
    return f"The price of the ticket to {destination_city} is {price}"


In [6]:
price_function ={
    "name": "ticket_price_query",
    "description": "Get the price of a return ticket to the destination city",
    "parameters":{
        "type": "object",
        "properties":{
            "destination_city":{
                "type":"string",
                "description" : "The city that the customer wants to tarvel to"
            },
        },
        "required": ["destination_city"],
        "additionalProperties":False

            
        }

    }

In [7]:
tools = [{"type":"function","function":price_function}]

In [8]:
tools

[{'type': 'function',
  'function': {'name': 'ticket_price_query',
   'description': 'Get the price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to tarvel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [ ]:
def handle_tool_call(message):
    tool_call  = message.tool_calls[0]
    if tool_call.function.name == "ticket_price_query":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = ticket_price_query(city)
        response = {
            "role":"tool",
            "content" : price_details,
            "tool_call_id" : tool_call.id
        }
    return response

In [10]:
Model = "llama3.1:latest"
def chat(message, history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model=Model, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response  = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model = Model, messages=messages)
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(fn = chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool Called for city : London
Tool Called for city : Paris
Tool Called for city : London
